In [1]:
import numpy as np
import matplotlib as plt
import pandas as pd

import qss
from utils.prep import prepare_2D_lattice_data

In [2]:
print(qss.DistType.FD)

DistType.FD


In [3]:
distros = [qss.DistType.FD, qss.DistType.FLB, qss.DistType.NFLB]
n =np.array([0.01] + [0.025 + 0.025 * x for x in range(39)] + [0.99])
betas= list(np.logspace(-2,3,61))
betas.append(0.2)
betas.append(np.inf)
betas = np.sort(betas)
betas = np.round(betas, 4)


In [7]:
N = 100
t = -1
t_prim = 0.25
# t_prim = 0

kx = np.linspace(-np.pi, np.pi, N, endpoint=False)
ky = np.linspace(-np.pi, np.pi, N, endpoint=False)
Ha =np.linspace(0,10, 100)

e_k, KX, KY = prepare_2D_lattice_data(kx, ky, t, t_prim)
n = np.round(n,3)
multi_index = pd.MultiIndex.from_product([distros, betas, Ha], names=["distributions", "beta", "H_a"])
mu_results= pd.DataFrame(columns=n, index=multi_index)
mu_results.columns = np.round(n.astype(np.float64),4)
mu_results

0.010 0.025 0.050 0.075 0.100 0.125 0.150 0.175  \
distributions beta H_a                                                        
DistType.FD   0.01 0.00000    NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
                   0.10101    NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
                   0.20202    NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
                   0.30303    NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
                   0.40404    NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
...                           ...   ...   ...   ...   ...   ...   ...   ...   
DistType.NFLB inf  9.59596    NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
                   9.69697    NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
                   9.79798    NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
                   9.89899    NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   
                   10.00000   NaN   NaN   NaN   NaN   NaN   NaN   NaN   NaN   

                            0.200 0.225  ... 0.775 0.800 0.825 0.850 0.875  \
distributions beta H_a                   ...                                 
DistType.FD   0.01 0.00000    NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   
                   0.10101    NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   
                   0.20202    NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   
                   0.30303    NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   
                   0.40404    NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   
...                           ...   ...  ...   ...   ...   ...   ...   ...   
DistType.NFLB inf  9.59596    NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   
                   9.69697    NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   
                   9.79798    NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   
                   9.89899    NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   
                   10.00000   NaN   NaN  ...   NaN   NaN   NaN   NaN   NaN   

                            0.900 0.925 0.950 0.975 0.990  
distributions beta H_a                                     
DistType.FD   0.01 0.00000    NaN   NaN   NaN   NaN   NaN  
                   0.10101    NaN   NaN   NaN   NaN   NaN  
                   0.20202    NaN   NaN   NaN   NaN   NaN  
                   0.30303    NaN   NaN   NaN   NaN   NaN  
                   0.40404    NaN   NaN   NaN   NaN   NaN  
...                           ...   ...   ...   ...   ...  
DistType.NFLB inf  9.59596    NaN   NaN   NaN   NaN   NaN  
                   9.69697    NaN   NaN   NaN   NaN   NaN  
                   9.79798    NaN   NaN   NaN   NaN   NaN  
                   9.89899    NaN   NaN   NaN   NaN   NaN  
                   10.00000   NaN   NaN   NaN   NaN   NaN  

[18900 rows x 41 columns]

In [8]:
mu_grid = qss.chemical_potential_B(mu_results, kx, ky, e_k)

TypeError: chemical_potential_B(): incompatible function arguments. The following argument types are supported:
    1. chemical_potential_B(dist: qss.DistType, beta: float, n_assumed: float, kx: ndarray[dtype=float64, order='C', writable=False], ky: ndarray[dtype=float64, order='C', writable=False], E: ndarray[dtype=float64, order='C', writable=False], H_a_T: float, g: float = 2.0, muB_eV_per_T: float = 5.788381806e-05, tol: float = 1e-06, max_iters: int = 300, expand_factor: float = 1.5, max_expands: int = 200, a0: float = -5000.0, b0: float = 1000.0) -> float

Invoked with types: pandas.core.frame.DataFrame, ndarray, ndarray, ndarray

In [ ]:
# Fallback: fill using the scalar API (compatible with current installed qss)
# This loops over all (dist, beta, H_a) rows and all n columns.
import time

def fill_mu_results_scalar(df, kx, ky, E):
    cols = df.columns.to_numpy(dtype=np.float64)
    out = np.empty((len(df.index), len(cols)), dtype=np.float64)
    for i, (dist, beta, Ha_val) in enumerate(df.index):
        for j, n_val in enumerate(cols):
            out[i, j] = qss.chemical_potential_B(
                dist, float(beta), float(n_val), kx, ky, E, float(Ha_val)
            )
    return out

_t0 = time.perf_counter()
mu_grid = fill_mu_results_scalar(mu_results, kx, ky, e_k)
_elapsed = time.perf_counter() - _t0
print(f"Computed mu_grid with scalar API in {_elapsed:.2f}s; shape = {mu_grid.shape}")


In [ ]:
# Fill the DataFrame with the computed mu values
mu_results.iloc[:, :] = mu_grid
print("mu_grid shape:", mu_grid.shape)
print("DataFrame shape:", mu_results.shape)
mu_results.head()

In [11]:
# Inspect one slice of the results for a specific (dist, beta, H_a)
idx = (qss.DistType.FD, float(betas[0]), float(Ha[0]))
mu_slice = mu_results.loc[idx]
mu_slice

0.010    NaN
0.025    NaN
0.050    NaN
0.075    NaN
0.100    NaN
0.125    NaN
0.150    NaN
0.175    NaN
0.200    NaN
0.225    NaN
0.250    NaN
0.275    NaN
0.300    NaN
0.325    NaN
0.350    NaN
0.375    NaN
0.400    NaN
0.425    NaN
0.450    NaN
0.475    NaN
0.500    NaN
0.525    NaN
0.550    NaN
0.575    NaN
0.600    NaN
0.625    NaN
0.650    NaN
0.675    NaN
0.700    NaN
0.725    NaN
0.750    NaN
0.775    NaN
0.800    NaN
0.825    NaN
0.850    NaN
0.875    NaN
0.900    NaN
0.925    NaN
0.950    NaN
0.975    NaN
0.990    NaN
Name: (DistType.FD, 0.01, 0.0), dtype: object

In [13]:
# Compute mu for every (dist, beta, H_a) row across all fillings n
mu_grid = qss.chemical_potential_B(mu_results, kx, ky, e_k)
mu_grid.shape

TypeError: chemical_potential_B(): incompatible function arguments. The following argument types are supported:
    1. chemical_potential_B(dist: qss.DistType, beta: float, n_assumed: float, kx: ndarray[dtype=float64, order='C', writable=False], ky: ndarray[dtype=float64, order='C', writable=False], E: ndarray[dtype=float64, order='C', writable=False], H_a_T: float, g: float = 2.0, muB_eV_per_T: float = 5.788381806e-05, tol: float = 1e-06, max_iters: int = 300, expand_factor: float = 1.5, max_expands: int = 200, a0: float = -5000.0, b0: float = 1000.0) -> float

Invoked with types: pandas.core.frame.DataFrame, ndarray, ndarray, ndarray

In [ ]:
# mu  = qss.chemical_potential_B
# mu(
#     qss.DistType.FD,
#     100,            # beta
#     0.5,            # n_assumed (filling)
#     kx,             # kx grid (1D)
#     ky,             # ky grid (1D)
#     e_k,            # band energies E(kx, ky)
#     0.0             # H_a/T (Zeeman term over temperature)
# )

-5000.0

In [14]:
# Optional: vectorized overload demo on a tiny subset to avoid long runtimes
# Returns an array of shape (len(distros_demo), len(betas_demo), len(n), len(Ha_demo))

distros_demo = [qss.DistType.FD]
betas_demo = np.array(betas[:1], dtype=np.float64)
Ha_demo = np.array(Ha[:1], dtype=np.float64)

mu_tensor_demo = qss.chemical_potential_B(
    distros_demo,
    betas_demo,        # betas (iterable or scalar)
    n,                 # fillings n (1-D array)
    kx,                # kx grid (1D)
    ky,                # ky grid (1D)
    e_k,               # band energies E(ky, kx)
    Ha_demo            # H_a values (iterable or scalar)
)
mu_tensor_demo.shape

TypeError: chemical_potential_B(): incompatible function arguments. The following argument types are supported:
    1. chemical_potential_B(dist: qss.DistType, beta: float, n_assumed: float, kx: ndarray[dtype=float64, order='C', writable=False], ky: ndarray[dtype=float64, order='C', writable=False], E: ndarray[dtype=float64, order='C', writable=False], H_a_T: float, g: float = 2.0, muB_eV_per_T: float = 5.788381806e-05, tol: float = 1e-06, max_iters: int = 300, expand_factor: float = 1.5, max_expands: int = 200, a0: float = -5000.0, b0: float = 1000.0) -> float

Invoked with types: list, ndarray, ndarray, ndarray, ndarray, ndarray, ndarray